# Store and product-family demand forecasting

This notebook forecasts sales separately for every store × product-family series in Kaggle Store Sales. It builds only past-demand lag/rolling features, uses promotion and holiday inputs only when known for the target dates, compares seasonal-naive with LightGBM and XGBoost when available, and evaluates identical rolling 16-day windows. The final two windows are held out from model selection.

In Kaggle settings choose GPU and two T4 accelerators if offered. The tree models train on CPU in this notebook; the extra T4 GPUs are not required for this workflow. The expected accuracy gains come from separate panel histories, causal lag features and covariates. Add Input: the Store Sales competition data. Outputs and model files are zipped at the end; click the link to download locally.

In [ ]:
# Cell 1: setup and configuration
import os, sys, gc, json, time, math, queue, zipfile, warnings, platform, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

QUICK = False            # True gives a fast smoke test
HORIZON = 30             # forecast horizon in days (7 and 14 are the first steps of the same forecast)
N_VAL_WINDOWS = 4        # rolling origin validation windows
N_TEST_WINDOWS = 2       # held out test windows, scored only after model selection
TRAIN_WINDOW_DAYS = 730  # training history length per fit
LOG_TARGET = True        # train on log1p(sales)
PARALLEL = None            # Automatically enables concurrent XGBoost jobs when two T4 GPUs are detected
LGB_TRY_GPU = False      # stock LightGBM on Kaggle usually has no GPU build
LAGS = [1, 7, 14, 28]
ROLL_WINDOWS = [7, 14, 28]

if QUICK:
    N_VAL_WINDOWS, N_TEST_WINDOWS, TRAIN_WINDOW_DAYS = 2, 1, 365

WORK = "/kaggle/working"
OUT = os.path.join(WORK, "forecast_outputs")
if os.path.isdir(OUT):
    shutil.rmtree(OUT)
os.makedirs(os.path.join(OUT, "figures"), exist_ok=True)
os.chdir(WORK)
RUN_START = time.time()

def minus(a, b):
    return np.subtract(a, b)

def secs_since(t0):
    return round(float(minus(time.time(), t0)), 1)

try:
    import torch
    N_GPUS = min(int(torch.cuda.device_count()), 2)
    GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPUS)]
    TORCH_VERSION = torch.__version__
except Exception:
    N_GPUS, GPU_NAMES, TORCH_VERSION = 0, [], None
CPU_COUNT = os.cpu_count() or 4
PARALLEL = N_GPUS >= 2

print("GPUs used:", N_GPUS, GPU_NAMES)
print("CPU cores:", CPU_COUNT)
print("xgboost", xgb.__version__, "lightgbm", lgb.__version__, "pandas", pd.__version__, "numpy", np.__version__)

In [ ]:
# Cell 2: load, validate, clean, build the series matrix and calendar
def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    return None

train_path = find_file("train.csv")
assert train_path is not None, "Attach the Store Sales competition data through Add Input"
base_dir = os.path.dirname(train_path)
train = pd.read_csv(train_path, parse_dates=["date"])
stores = pd.read_csv(os.path.join(base_dir, "stores.csv"))
hol_path = os.path.join(base_dir, "holidays_events.csv")
holidays = pd.read_csv(hol_path, parse_dates=["date"]) if os.path.exists(hol_path) else None

cleaning_log = {
    "source_path": train_path,
    "retrieval_date": time.strftime("%Y%m%d"),
    "terms_note": "The Kaggle data page marks this data subject to Competition Rules. Review those rules before redistributing derived data; raw data is not packaged.",
    "rows": int(len(train)),
    "columns": list(train.columns),
    "date_min": str(train.date.min().date()),
    "date_max": str(train.date.max().date()),
    "exact_duplicate_rows": int(train.duplicated().sum()),
    "duplicate_date_store_family_keys": int(train.duplicated(["date", "store_nbr", "family"]).sum()),
    "negative_sales_rows": int((train.sales < 0).sum()),
    "null_counts": {k: int(v) for k, v in train.isna().sum().to_dict().items()},
}

full_dates = pd.date_range(train.date.min(), train.date.max(), freq="D")
missing_dates = full_dates.difference(pd.DatetimeIndex(train.date.unique()))
cleaning_log["missing_calendar_dates"] = [str(d.date()) for d in missing_dates]
cleaning_log["missing_date_decision"] = (
    "Dates absent for every store (for example Dec 25) are treated as closed days and filled with 0 sales."
)

piv = train.pivot_table(index="date", columns=["store_nbr", "family"], values="sales", aggfunc="sum")
piv = piv.reindex(full_dates)
cleaning_log["missing_cells_filled_with_zero"] = int(piv.isna().sum().sum())
piv = piv.fillna(0.0)
S = piv.values.astype(np.float32)          # raw sales, shape (days, series)
N_HIST, NS = S.shape
cleaning_log["series_count"] = int(NS)
cleaning_log["history_days"] = int(N_HIST)
cleaning_log["outlier_policy"] = "Outliers are flagged for investigation, not deleted."

meta = pd.DataFrame(list(piv.columns), columns=["store_nbr", "family"]).merge(stores, on="store_nbr", how="left")
fam_codes = pd.Categorical(meta.family).codes
type_codes = pd.Categorical(meta["type"]).codes
STATIC = np.column_stack([meta.store_nbr.values, fam_codes, meta.cluster.values, type_codes]).astype(np.float32)
NSTAT = STATIC.shape[1]

# calendar over history plus the future horizon (only features known in advance)
all_dates = full_dates.append(pd.date_range(start=full_dates.max() + pd.Timedelta(days=1), periods=HORIZON, freq="D"))
N_TOTAL = len(all_dates)
if holidays is not None:
    hol = holidays[(holidays.locale == "National") & (~holidays.transferred) & (holidays.type != "Work Day")]
    hol_set = set(hol.date.dt.normalize())
else:
    hol_set = set()

cal = pd.DataFrame({
    "dow": all_dates.dayofweek,
    "dom": all_dates.day,
    "month": all_dates.month,
    "quarter": all_dates.quarter,
    "year": all_dates.year,
    "is_weekend": (all_dates.dayofweek >= 5).astype(int),
    "is_payday": ((all_dates.day == 15) | all_dates.is_month_end).astype(int),
    "national_holiday": [int(d in hol_set) for d in all_dates],
})
CAL_NAMES = list(cal.columns)
CAL = cal.values.astype(np.float32)
NCAL = CAL.shape[1]

def to_space(x):
    return np.log1p(x) if LOG_TARGET else x

def from_space(x):
    return np.expm1(x) if LOG_TARGET else x

M = np.zeros((N_TOTAL, NS), dtype=np.float32)
M[:N_HIST] = to_space(S)

FEATURE_NAMES = (CAL_NAMES + ["store_nbr", "family_code", "cluster", "store_type_code"]
                 + [f"lag_{k}" for k in LAGS]
                 + sum([[f"roll_mean_{w}", f"roll_std_{w}"] for w in ROLL_WINDOWS], []))
NFEAT = len(FEATURE_NAMES)
FIRST = max(LAGS + ROLL_WINDOWS)

def day_features(Mx, d):
    # uses only rows strictly before day d, so the same code is safe in training and recursive forecasting
    out = np.empty((NS, NFEAT), dtype=np.float32)
    c = 0
    out[:, c:c + NCAL] = CAL[d]
    c += NCAL
    out[:, c:c + NSTAT] = STATIC
    c += NSTAT
    for k in LAGS:
        out[:, c] = Mx[int(minus(d, k))]
        c += 1
    for w in ROLL_WINDOWS:
        blk = Mx[int(minus(d, w)):d]
        out[:, c] = blk.mean(axis=0)
        out[:, c + 1] = blk.std(axis=0)
        c += 2
    return out

n_feat_days = int(minus(N_HIST, FIRST))
X_all = np.empty((n_feat_days, NS, NFEAT), dtype=np.float32)
for i in range(n_feat_days):
    X_all[i] = day_features(M, FIRST + i)
Y_all = M[FIRST:N_HIST].copy()

print("series:", NS, "history days:", N_HIST, "features:", NFEAT)
print("missing calendar dates filled with 0:", cleaning_log["missing_calendar_dates"])
print("feature array MB:", round(X_all.nbytes / 1e6, 1))

In [ ]:
# Cell 2b: exploratory analysis, decomposition, and cleaning evidence
# All charts are computed from observed training history only. Flagged values are retained.
import calendar as calendar_lib

daily_total = pd.Series(S.sum(axis=1), index=full_dates, name="sales")
eda_summary = {
    "business_question": "How does daily grocery demand vary over time, weekday, month, store, and product family?",
    "rows": int(len(train)), "series_count": int(NS), "date_start": str(full_dates.min().date()),
    "date_end": str(full_dates.max().date()), "target_units": "sales units per date/store/family",
    "sales_summary": {k: float(v) for k, v in train.sales.describe().to_dict().items()},
    "zero_sales_rows": int((train.sales == 0).sum()), "zero_sales_share": float((train.sales == 0).mean()),
    "positive_sales_rows": int((train.sales > 0).sum()), "negative_sales_rows": int((train.sales < 0).sum()),
    "store_count": int(train.store_nbr.nunique()), "family_count": int(train.family.nunique()),
    "weekday_mean_units": {str(k): float(v) for k, v in daily_total.groupby(daily_total.index.day_name()).mean().items()},
    "month_mean_units": {str(int(k)): float(v) for k, v in daily_total.groupby(daily_total.index.month).mean().items()},
    "year_mean_units": {str(int(k)): float(v) for k, v in daily_total.groupby(daily_total.index.year).mean().items()},
    "outlier_rule": "1.5 IQR flags on daily aggregate only; flags are investigated and never automatically deleted.",
    "decomposition": "Additive weekly decomposition: daily totals include true zero-sales closure dates, so multiplicative decomposition is undefined; decomposition is diagnostic, not a model assumption.",
    "stationarity": "Not applicable: the selected approach is a recursive boosted-tree model, not an ARIMA-family model.",
}
fig_dir = os.path.join(OUT, "figures")
os.makedirs(fig_dir, exist_ok=True)

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(daily_total.index, daily_total, color="#7892a5", alpha=.45, linewidth=.7, label="Daily total")
ax.plot(daily_total.index, daily_total.rolling(28, min_periods=1).mean(), color="#c66b45", linewidth=1.7, label="28-day rolling mean")
ax.set(title="Daily grocery demand and 28-day rolling mean", ylabel="Units per day (all stores and families)")
ax.legend(); fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "eda_demand_trend.png"), dpi=130); plt.close(fig)

by_weekday = daily_total.groupby(daily_total.index.dayofweek).mean().reindex(range(7))
by_month = daily_total.groupby(daily_total.index.month).mean().reindex(range(1, 13))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([calendar_lib.day_name[i] for i in range(7)], by_weekday.values, color="#789b82")
axes[0].set(title="Average daily demand by weekday", ylabel="Mean units")
axes[0].tick_params(axis="x", rotation=35)
axes[1].bar([calendar_lib.month_abbr[i] for i in range(1, 13)], by_month.values, color="#c69a53")
axes[1].set(title="Average daily demand by month", ylabel="Mean units")
fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "eda_calendar_patterns.png"), dpi=130); plt.close(fig)

by_year = daily_total.groupby(daily_total.index.year).mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(by_year.index.astype(str), by_year.values, marker="o", color="#6487a0")
ax.set(title="Average daily demand by year", ylabel="Mean units per day")
fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "eda_yearly_demand.png"), dpi=130); plt.close(fig)

family_totals = train.groupby("family").sales.sum().sort_values(ascending=False)
store_totals = train.groupby("store_nbr").sales.sum().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
family_totals.head(12).sort_values().plot.barh(ax=axes[0], color="#789b82")
axes[0].set(title="Highest-volume product families", xlabel="Observed units")
store_totals.sort_values().plot.barh(ax=axes[1], color="#c69a53")
axes[1].set(title="Store totals", xlabel="Observed units")
fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "eda_store_family_comparison.png"), dpi=130); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train.sales.clip(upper=train.sales.quantile(.995)), bins=70, color="#6487a0")
axes[0].set(title="Sales distribution (display clipped at 99.5th percentile)", xlabel="Units")
weekday_table = pd.DataFrame({"sales": daily_total.to_numpy(), "month": full_dates.month, "weekday": full_dates.dayofweek})
heat = weekday_table.pivot_table(index="month", columns="weekday", values="sales", aggfunc="mean").reindex(index=range(1, 13), columns=range(7))
axes[1].imshow(heat, aspect="auto", cmap="YlGnBu")
axes[1].set(title="Mean daily demand calendar heatmap", xlabel="Weekday (Mon=0)", ylabel="Month")
fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "eda_distribution_calendar_heatmap.png"), dpi=130); plt.close(fig)

q1, q3 = daily_total.quantile([.25, .75]); iqr = q3 - q1
flagged = daily_total[(daily_total < q1 - 1.5 * iqr) | (daily_total > q3 + 1.5 * iqr)].rename("daily_units").reset_index(names="date")
flagged.to_csv(os.path.join(OUT, "eda_daily_outlier_flags.csv"), index=False)
eda_summary["daily_aggregate_iqr_flags"] = int(len(flagged))
eda_summary["largest_aggregate_days"] = [{"date": str(k.date()), "units": float(v)} for k, v in daily_total.nlargest(10).items()]
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    decomposition = seasonal_decompose(daily_total, model="additive", period=7, extrapolate_trend="freq")
    fig = decomposition.plot(); fig.set_size_inches(13, 8); fig.tight_layout()
    fig.savefig(os.path.join(fig_dir, "eda_weekly_decomposition.png"), dpi=130); plt.close(fig)
    eda_summary["decomposition_status"] = "Completed additive weekly decomposition of total daily demand."
except Exception as exc:
    eda_summary["decomposition_status"] = f"Could not run optional decomposition: {type(exc).__name__}: {exc}"

with open(os.path.join(OUT, "eda_summary.json"), "w") as f:
    json.dump(eda_summary, f, indent=2, default=str)
print("EDA summary:", json.dumps({k: eda_summary[k] for k in ["rows", "series_count", "date_start", "date_end", "zero_sales_share", "daily_aggregate_iqr_flags", "decomposition_status"]}, indent=2))
print("Interpretation: store and family totals are highly unequal, supporting a pooled model with entity identifiers. Weekly patterns motivate weekday and weekly-lag features; aggregate spikes are retained and reported for review.")


In [3]:
# Cell 3: metrics, baselines, rolling origin windows
def smape_pct(a, f):
    den = np.abs(a) + np.abs(f)
    num = 2.0 * np.abs(minus(a, f))
    safe = np.where(den == 0, 1.0, den)
    return float(100.0 * np.where(den == 0, 0.0, num / safe).mean())

def metrics(a, f):
    a = np.asarray(a, dtype=np.float64)
    f = np.asarray(f, dtype=np.float64)
    err = minus(a, f)
    ss_res = float((err ** 2).sum())
    ss_tot = float((minus(a, a.mean()) ** 2).sum())
    return {
        "MAE": float(np.abs(err).mean()),
        "RMSE": float(np.sqrt((err ** 2).mean())),
        "sMAPE_pct": smape_pct(a, f),
        "RMSLE": float(np.sqrt((minus(np.log1p(a), np.log1p(np.maximum(f, 0.0))) ** 2).mean())),
        "R2": float(minus(1.0, ss_res / ss_tot)) if ss_tot > 0 else float("nan"),
    }

def naive_forecast(origin):
    return np.tile(S[origin], (HORIZON, 1))

def seasonal_naive_forecast(origin):
    base = int(minus(origin, 6))
    idx = [base + (k % 7) for k in range(HORIZON)]
    return S[idx]

LAST = int(minus(N_HIST, 1))
N_WIN = N_VAL_WINDOWS + N_TEST_WINDOWS
windows = []
for j in range(N_WIN):
    blocks_back = int(minus(N_WIN, j))
    start = int(minus(N_HIST, blocks_back * HORIZON))
    windows.append({
        "idx": j,
        "role": "val" if j < N_VAL_WINDOWS else "test",
        "origin": int(minus(start, 1)),
        "start": start,
        "end": start + HORIZON,
    })
    assert windows[j]["origin"] > FIRST + 100

for w in windows:
    print(w["idx"], w["role"], "origin", str(all_dates[w["origin"]].date()),
          "target", str(all_dates[w["start"]].date()), "to", str(all_dates[int(minus(w["end"], 1))].date()))

0 val origin 2017-02-16 target 2017-02-17 to 2017-03-18
1 val origin 2017-03-18 target 2017-03-19 to 2017-04-17
2 val origin 2017-04-17 target 2017-04-18 to 2017-05-17
3 val origin 2017-05-17 target 2017-05-18 to 2017-06-16
4 test origin 2017-06-16 target 2017-06-17 to 2017-07-16
5 test origin 2017-07-16 target 2017-07-17 to 2017-08-15


In [4]:
# Cell 4: training, recursive forecasting, parallel execution across two GPUs
XGB_PARAMS = dict(tree_method="hist", learning_rate=0.05, max_depth=8, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0,
                  objective="reg:squarederror", random_state=SEED, n_jobs=CPU_COUNT)
LGB_PARAMS = dict(learning_rate=0.05, num_leaves=127, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, min_child_samples=50, reg_lambda=1.0,
                  random_state=SEED, n_jobs=CPU_COUNT, verbosity=0)
MAX_TREES = 2000
EARLY_STOP = 100

def window_slices(origin):
    lo = max(FIRST, int(minus(origin, TRAIN_WINDOW_DAYS)) + 1)
    a = int(minus(lo, FIRST))
    b = int(minus(origin, FIRST)) + 1
    return X_all[a:b], Y_all[a:b]

def fit_model(kind, origin, gpu_id):
    Xs, Ys = window_slices(origin)
    nd = Xs.shape[0]
    cut = int(minus(nd, HORIZON))
    Xtr = Xs[:cut].reshape(cut * NS, NFEAT)
    ytr = Ys[:cut].reshape(cut * NS)
    Xva = Xs[cut:].reshape(HORIZON * NS, NFEAT)
    yva = Ys[cut:].reshape(HORIZON * NS)
    Xall = Xs.reshape(nd * NS, NFEAT)
    yall = Ys.reshape(nd * NS)
    t0 = time.time()
    info = {"kind": kind, "origin": origin, "train_rows": int(nd * NS)}
    if kind == "xgboost":
        dev = f"cuda:{gpu_id}" if gpu_id is not None else "cpu"
        try:
            m = xgb.XGBRegressor(device=dev, n_estimators=MAX_TREES, early_stopping_rounds=EARLY_STOP, **XGB_PARAMS)
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        except Exception as e:
            print("XGBoost GPU failed, using CPU:", str(e)[:120])
            dev = "cpu"
            m = xgb.XGBRegressor(device=dev, n_estimators=MAX_TREES, early_stopping_rounds=EARLY_STOP, **XGB_PARAMS)
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        best = int(m.best_iteration) + 1
        final = xgb.XGBRegressor(device=dev, n_estimators=best, **XGB_PARAMS)
        final.fit(Xall, yall, verbose=False)
        final.get_booster().set_param({"device": "cpu"})  # predict on CPU, avoids the device mismatch
        info.update({"device": dev, "best_iteration": best})
    else:
        def run(device_type):
            p = dict(LGB_PARAMS)
            p["device_type"] = device_type
            m = lgb.LGBMRegressor(n_estimators=MAX_TREES, **p)
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)],
                  callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
            best = int(m.best_iteration_ or MAX_TREES)
            f = lgb.LGBMRegressor(n_estimators=best, **p)
            f.fit(Xall, yall)
            return f, best
        dev = "cpu"
        if LGB_TRY_GPU:
            try:
                final, best = run("gpu")
                dev = "gpu"
            except Exception as e:
                print("LightGBM GPU unavailable, using CPU:", str(e)[:120])
                final, best = run("cpu")
        else:
            final, best = run("cpu")
        info.update({"device": dev, "best_iteration": best})
    info["seconds"] = secs_since(t0)
    return final, info

def recursive_forecast(model, origin):
    # direct use of earlier predictions as lags: no actual value after the origin is ever read
    W = M.copy()
    W[int(origin) + 1:] = 0.0
    out = np.zeros((HORIZON, NS), dtype=np.float32)
    for k in range(HORIZON):
        d = int(origin) + 1 + k
        p = np.maximum(model.predict(day_features(W, d)), 0.0)
        W[d] = p
        out[k] = p
    return out

def run_window_job(kind, widx, gpu_id):
    w = windows[widx]
    model, info = fit_model(kind, w["origin"], gpu_id)
    pred_space = recursive_forecast(model, w["origin"])
    info["window"] = widx
    print(f"done {kind} window {widx} on {info['device']} in {info['seconds']}s, trees {info['best_iteration']}")
    return kind, widx, pred_space, info

gpu_q = queue.Queue()
for g in range(max(N_GPUS, 1)):
    gpu_q.put(g if N_GPUS > 0 else None)

def xgb_job(widx):
    g = gpu_q.get()
    try:
        return run_window_job("xgboost", widx, g)
    finally:
        gpu_q.put(g)

def lgb_job(widx):
    return run_window_job("lightgbm", widx, None)

def run_role(role):
    idxs = [i for i, w in enumerate(windows) if w["role"] == role]
    results = {}
    if PARALLEL:
        with ThreadPoolExecutor(max_workers=max(N_GPUS, 1)) as px, ThreadPoolExecutor(max_workers=1) as pl:
            futs = [px.submit(xgb_job, i) for i in idxs] + [pl.submit(lgb_job, i) for i in idxs]
            for f in futs:
                kind, widx, pred, info = f.result()
                results[(kind, widx)] = (pred, info)
    else:
        for i in idxs:
            for kind in ["xgboost", "lightgbm"]:
                g = 0 if (kind == "xgboost" and N_GPUS > 0) else None
                k, widx, pred, info = run_window_job(kind, i, g)
                results[(k, widx)] = (pred, info)
    return results

def score_role(role, results):
    rows, store = [], {}
    for widx, w in enumerate(windows):
        if w["role"] != role:
            continue
        actual = S[w["start"]:w["end"]]
        cand = {"naive": naive_forecast(w["origin"]),
                "seasonal_naive": seasonal_naive_forecast(w["origin"])}
        for kind in ["lightgbm", "xgboost"]:
            cand[kind] = np.maximum(from_space(results[(kind, widx)][0]), 0.0)
        store[widx] = dict(actual=actual, **cand)
        for name, fc in cand.items():
            m = metrics(actual, fc)
            m.update({"model": name, "window": widx, "role": role})
            rows.append(m)
    return pd.DataFrame(rows), store

t0 = time.time()
val_res = run_role("val")
val_df, val_store = score_role("val", val_res)
print("validation elapsed seconds:", secs_since(t0))
cols = ["MAE", "RMSE", "sMAPE_pct", "RMSLE", "R2"]
print(val_df[["window", "model"] + cols].round(3).to_string(index=False))
val_mean = val_df.groupby("model")[cols].mean()
print("\nMean over validation windows")
print(val_mean.round(3))

done xgboost window 0 on cuda:0 in 9.8s, trees 206
done lightgbm window 0 on cpu in 96.8s, trees 762
done xgboost window 1 on cuda:0 in 24.1s, trees 842
done lightgbm window 1 on cpu in 78.2s, trees 555
done xgboost window 2 on cuda:0 in 21.2s, trees 700
done lightgbm window 2 on cpu in 88.5s, trees 660
done xgboost window 3 on cuda:0 in 13.2s, trees 357
done lightgbm window 3 on cpu in 98.5s, trees 756
validation elapsed seconds: 437.2
 window          model     MAE    RMSE  sMAPE_pct  RMSLE    R2
      0          naive 141.063 554.277     51.239  0.645 0.833
      0 seasonal_naive  89.485 342.475     43.894  0.600 0.936
      0       lightgbm  73.611 296.723     60.066  0.475 0.952
      0        xgboost  75.080 318.459     52.375  0.474 0.945
      1          naive 169.238 649.467     51.800  0.717 0.789
      1 seasonal_naive  83.671 364.721     43.814  0.629 0.933
      1       lightgbm  75.472 342.185     60.494  0.543 0.941
      1        xgboost  77.487 357.810     53.347  0.54

In [5]:
# Cell 5: select on validation only, then score the untouched test windows
ml = val_mean.loc[["lightgbm", "xgboost"]].copy()
ml["rank_score"] = ml[["MAE", "RMSE", "sMAPE_pct"]].rank().mean(axis=1)
order = ml.sort_values(["rank_score", "MAE"])
SELECTED = order.index[0]
base_mae = float(val_mean.loc["seasonal_naive", "MAE"])
print("Selected model (mean rank of MAE, RMSE, sMAPE on validation):", SELECTED)
print("Beats seasonal naive on validation MAE:", bool(ml.loc[SELECTED, "MAE"] < base_mae))
print("Beats seasonal naive on validation sMAPE:", bool(ml.loc[SELECTED, "sMAPE_pct"] < float(val_mean.loc["seasonal_naive", "sMAPE_pct"])))

t0 = time.time()
test_res = run_role("test")
test_df, test_store = score_role("test", test_res)
print("test elapsed seconds:", secs_since(t0))
print(test_df[["window", "model"] + cols].round(3).to_string(index=False))
test_mean = test_df.groupby("model")[cols].mean()
print("\nMean over test windows")
print(test_mean.round(3))

Selected model (mean rank of MAE, RMSE, sMAPE on validation): lightgbm
Beats seasonal naive on validation MAE: True
Beats seasonal naive on validation sMAPE: False
done xgboost window 4 on cuda:0 in 25.1s, trees 857
done lightgbm window 4 on cpu in 143.0s, trees 1220
done xgboost window 5 on cuda:0 in 15.9s, trees 463
done lightgbm window 5 on cpu in 65.3s, trees 428
test elapsed seconds: 253.4
 window          model     MAE    RMSE  sMAPE_pct  RMSLE    R2
      4          naive 138.425 467.202     49.958  0.619 0.880
      4 seasonal_naive  78.099 302.292     41.370  0.520 0.950
      4       lightgbm  59.529 241.492     54.749  0.401 0.968
      4        xgboost  64.125 260.490     50.548  0.404 0.963
      5          naive 198.937 691.061     51.420  0.665 0.700
      5 seasonal_naive  79.689 316.893     41.777  0.585 0.937
      5       lightgbm  66.109 240.145     56.930  0.474 0.964
      5        xgboost  67.489 252.141     54.259  0.480 0.960

Mean over test windows
           

In [ ]:
# Cell 6: final model on all history, 30 day forecast, uncertainty, insights
t0 = time.time()
final_gpu = 0 if N_GPUS > 0 else None
final_model, final_info = fit_model(SELECTED, LAST, final_gpu)
final_pred_space = recursive_forecast(final_model, LAST)
print("final fit seconds:", secs_since(t0), final_info)

# uncertainty: empirical 10th and 90th percentile of validation residuals per horizon step
res_list = []
for widx, w in enumerate(windows):
    if w["role"] == "val":
        pred_space = val_res[(SELECTED, widx)][0]
        res_list.append(minus(M[w["start"]:w["end"]], pred_space))
res_all = np.concatenate(res_list, axis=1)
q_lo = np.quantile(res_all, 0.1, axis=1)
q_hi = np.quantile(res_all, 0.9, axis=1)

point = np.maximum(from_space(final_pred_space), 0.0)
lower = np.maximum(from_space(final_pred_space + q_lo[:, None]), 0.0)
upper = np.maximum(from_space(final_pred_space + q_hi[:, None]), 0.0)

future_dates = all_dates[N_HIST:N_TOTAL]
forecast_df = pd.DataFrame({
    "date": np.repeat(future_dates.values, NS),
    "horizon_day": np.repeat(np.arange(1, HORIZON + 1), NS),
    "store_nbr": np.tile(meta.store_nbr.values, HORIZON),
    "family": np.tile(meta.family.values, HORIZON),
    "forecast": point.reshape(HORIZON * NS),
    "lower_p10": lower.reshape(HORIZON * NS),
    "upper_p90": upper.reshape(HORIZON * NS),
})
forecast_df["model"] = SELECTED
forecast_df["interval_method"] = "empirical validation residual quantiles per horizon step, pooled across series"

totals = pd.DataFrame({"date": future_dates, "horizon_day": np.arange(1, HORIZON + 1), "total_forecast_units": point.sum(axis=1)})
horizon_summary = pd.DataFrame([{"horizon_days": h, "total_forecast_units": float(point[:h].sum())} for h in [7, 14, 30]])

# insights tied to forecast numbers
last_block = S[int(minus(N_HIST, HORIZON)):N_HIST]
fam_df = pd.DataFrame({"family": meta.family.values, "last30_actual": last_block.sum(axis=0), "next30_forecast": point.sum(axis=0)})
fam_df = fam_df.groupby("family", as_index=False).sum()
fam_df["change_pct"] = 100.0 * minus(fam_df.next30_forecast, fam_df.last30_actual) / fam_df.last30_actual.replace(0, np.nan)
fam_df = fam_df.sort_values("change_pct", ascending=False)

recent = S[int(minus(N_HIST, 90)):N_HIST]
vol = pd.DataFrame({"store_nbr": meta.store_nbr.values, "family": meta.family.values,
                    "mean90": recent.mean(axis=0), "std90": recent.std(axis=0)})
vol["cv90"] = vol.std90 / vol.mean90.replace(0, np.nan)
volatile = vol[vol.mean90 > 1.0].sort_values("cv90", ascending=False).head(25)

peak_day = totals.loc[totals.total_forecast_units.idxmax()]
insights = {
    "model_used": SELECTED,
    "expected_peak_day": str(peak_day.date.date()),
    "expected_peak_units": float(peak_day.total_forecast_units),
    "top_growth_families": fam_df.head(3)[["family", "change_pct"]].to_dict("records"),
    "top_decline_families": fam_df.tail(3)[["family", "change_pct"]].to_dict("records"),
    "lowest_forecast_families": fam_df.nsmallest(3, "next30_forecast")[["family", "next30_forecast"]].to_dict("records"),
    "evidence_files": ["family_outlook.csv", "volatile_series.csv", "forecast_30d.csv"],
    "unavailable_inputs": ["stock levels", "supplier lead times"],
    "note": "No stockout or inventory risk is claimed because inventory and lead time inputs are not in this dataset.",
}
print(horizon_summary)
print(fam_df.head(5).round(2))
print(fam_df.tail(5).round(2))

In [ ]:
# Cell 7: error analysis, charts, and all exports
# per window predictions (actual and every model)
frames = []
for role, store in [("val", val_store), ("test", test_store)]:
    for widx, d in store.items():
        w = windows[widx]
        n = HORIZON * NS
        fr = pd.DataFrame({
            "window": widx, "role": role,
            "date": np.repeat(all_dates[w["start"]:w["end"]].values, NS),
            "store_nbr": np.tile(meta.store_nbr.values, HORIZON),
            "family": np.tile(meta.family.values, HORIZON),
            "actual": d["actual"].reshape(n),
            "naive": d["naive"].reshape(n),
            "seasonal_naive": d["seasonal_naive"].reshape(n),
            "lightgbm": d["lightgbm"].reshape(n),
            "xgboost": d["xgboost"].reshape(n),
        })
        frames.append(fr)
pred_df = pd.concat(frames, ignore_index=True)

# largest misses of the selected model
misses = pred_df.copy()
misses["abs_error"] = np.abs(minus(misses.actual, misses[SELECTED]))
top_misses = (misses.groupby(["role", "window", "store_nbr", "family"], as_index=False)
              .agg(total_abs_error=("abs_error", "sum"), actual_sum=("actual", "sum"), forecast_sum=(SELECTED, "sum"))
              .sort_values("total_abs_error", ascending=False).head(40))

# charts
fig_dir = os.path.join(OUT, "figures")
sel_windows = [i for i, w in enumerate(windows) if w["role"] == "test"]
fig, ax = plt.subplots(figsize=(13, 4.5))
for widx in sel_windows:
    w = windows[widx]
    d = (test_store[widx])
    x = all_dates[w["start"]:w["end"]]
    ax.plot(x, d["actual"].sum(axis=1), color="#1f77b4", label="Actual" if widx == sel_windows[0] else None)
    ax.plot(x, d[SELECTED].sum(axis=1), color="#d62728", label=f"{SELECTED}" if widx == sel_windows[0] else None)
    ax.plot(x, d["seasonal_naive"].sum(axis=1), color="#7f7f7f", linestyle="dashed", label="Seasonal naive" if widx == sel_windows[0] else None)
ax.set_title("Test windows: total daily units, actual vs forecast")
ax.set_ylabel("Units per day, all series")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(fig_dir, "test_actual_vs_forecast.png"), dpi=130)
plt.close(fig)

fig, ax = plt.subplots(figsize=(13, 4))
for role, store in [("val", val_store), ("test", test_store)]:
    for widx, d in store.items():
        w = windows[widx]
        x = all_dates[w["start"]:w["end"]]
        ax.plot(x, minus(d["actual"].sum(axis=1), d[SELECTED].sum(axis=1)), color="#2ca02c" if role == "val" else "#ff7f0e")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Residuals over time (total units, actual minus forecast), green validation, orange test")
fig.tight_layout()
fig.savefig(os.path.join(fig_dir, "residuals_over_time.png"), dpi=130)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(res_all.reshape(res_all.shape[0] * res_all.shape[1]), bins=80, color="#9467bd")
ax.set_title("Validation residual distribution (model space)")
fig.tight_layout()
fig.savefig(os.path.join(fig_dir, "residual_distribution.png"), dpi=130)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
val_mean["MAE"].plot(kind="bar", ax=ax, color="#1f77b4")
ax.set_title("Mean validation MAE by model")
fig.tight_layout()
fig.savefig(os.path.join(fig_dir, "validation_mae_by_model.png"), dpi=130)
plt.close(fig)

# Save the serving mappings and calibrated interval quantiles required for fresh API predictions.
series_export = meta[["store_nbr", "family", "cluster"]].copy()
series_export["family_code"] = fam_codes
series_export["store_type_code"] = type_codes
series_export["series_index"] = np.arange(NS, dtype=int)
series_export.to_csv(os.path.join(OUT, "series_metadata.csv"), index=False)
inference_metadata = {
    "selected_model": SELECTED, "features": FEATURE_NAMES, "calendar_features": CAL_NAMES,
    "log_target": LOG_TARGET, "minimum_history_days": 28,
    "national_holiday_dates": sorted(str(d.date()) for d in hol_set),
    "residual_q10_by_horizon": [float(v) for v in q_lo],
    "residual_q90_by_horizon": [float(v) for v in q_hi],
    "interval_method": "Empirical validation residual quantiles by horizon step, pooled across series; calibrated in model space.",
    "input_schema": ["date", "store_nbr", "family", "sales"],
}
with open(os.path.join(OUT, "inference_metadata.json"), "w") as f:
    json.dump(inference_metadata, f, indent=2, default=str)

# exports
val_df.to_csv(os.path.join(OUT, "validation_metrics_by_window.csv"), index=False)
val_mean.to_csv(os.path.join(OUT, "validation_metrics_mean.csv"))
test_df.to_csv(os.path.join(OUT, "test_metrics_by_window.csv"), index=False)
test_mean.to_csv(os.path.join(OUT, "test_metrics_mean.csv"))
pred_df.to_csv(os.path.join(OUT, "window_predictions.csv"), index=False)
top_misses.to_csv(os.path.join(OUT, "largest_misses.csv"), index=False)
forecast_df.to_csv(os.path.join(OUT, "forecast_30d.csv"), index=False)
totals.to_csv(os.path.join(OUT, "forecast_totals_daily.csv"), index=False)
horizon_summary.to_csv(os.path.join(OUT, "horizon_summary.csv"), index=False)
fam_df.to_csv(os.path.join(OUT, "family_outlook.csv"), index=False)
volatile.to_csv(os.path.join(OUT, "volatile_series.csv"), index=False)

if SELECTED == "xgboost":
    final_model.save_model(os.path.join(OUT, "selected_xgboost.json"))
else:
    final_model.booster_.save_model(os.path.join(OUT, "selected_lightgbm.txt"))

data_dictionary = {
    "source_train.csv": {"id": "source row identifier", "date": "daily observation date", "store_nbr": "store identifier",
                         "family": "product family", "sales": "daily target in sales units", "onpromotion": "observed promotion count; excluded because future values are unavailable"},
    "source_stores.csv": {"store_nbr": "store identifier", "city": "store city (not used in model)", "state": "store state (not used in model)",
                           "type": "store class encoded as store_type_code", "cluster": "store cluster used as a static model feature"},
    "source_holidays_events.csv": {"date": "calendar event date", "type": "event category", "locale": "event scope", "transferred": "whether event was moved; transferred events excluded from holiday feature"},
    "model_features": {"dow": "weekday, Monday=0", "dom": "day of month", "month": "calendar month", "quarter": "calendar quarter", "year": "calendar year",
                       "is_weekend": "Saturday/Sunday flag", "is_payday": "day 15 or month-end flag", "national_holiday": "known national-holiday flag",
                       "store_nbr": "store id", "family_code": "categorical product-family code", "cluster": "store cluster", "store_type_code": "categorical store-type code",
                       "lag_1/7/14/28": "sales-space values from prior days", "roll_mean/std_7/14/28": "past-only rolling statistics; no current target is included"},
    "forecast_30d.csv": {"date": "forecast date", "horizon_day": "1 to 30 days after last observed date", "store_nbr": "store id",
                         "family": "product family", "forecast": "predicted units", "lower_p10": "10th percentile interval bound", "upper_p90": "90th percentile interval bound"},
    "window_predictions.csv": "actual and model predictions for each chronological validation and held-out test window",
    "validation_metrics_*.csv": "MAE, RMSE, sMAPE percent, RMSLE and R2 on validation windows",
    "test_metrics_*.csv": "same metrics on held-out test windows",
    "family_outlook.csv": "previous 30 actual days versus next 30 forecast days by family",
    "series_metadata.csv": "series order and static categorical codes required by the inference service",
    "inference_metadata.json": "ordered model features, calendar definitions and calibrated interval quantiles",
}
run_meta = {
    "approach": "one pooled model over all store and family series, recursive multi step forecast, log1p target" if LOG_TARGET else "one pooled model, recursive multi step forecast, raw target",
    "selected_model": SELECTED,
    "selection_rule": "mean rank of MAE, RMSE, sMAPE on validation windows only",
    "windows": [{"idx": w["idx"], "role": w["role"], "origin": str(all_dates[w["origin"]].date()),
                 "target_start": str(all_dates[w["start"]].date()),
                 "target_end": str(all_dates[int(minus(w["end"], 1))].date())} for w in windows],
    "features": FEATURE_NAMES,
    "excluded_features": "onpromotion is excluded because it is unknown for the future horizon",
    "caveats": ["year is a numeric feature and trees cannot extrapolate it", "recursive forecasting accumulates error with horizon"],
    "seed": SEED, "horizon": HORIZON, "train_window_days": TRAIN_WINDOW_DAYS,
    "xgb_params": XGB_PARAMS, "lgb_params": LGB_PARAMS, "max_trees": MAX_TREES, "early_stopping_rounds": EARLY_STOP,
    "validation_fit_info": {f"{k[0]}_{k[1]}": v[1] for k, v in val_res.items()},
    "test_fit_info": {f"{k[0]}_{k[1]}": v[1] for k, v in test_res.items()},
    "final_fit_info": final_info,
    "hardware": {"gpus": GPU_NAMES, "cpu_count": CPU_COUNT, "platform": platform.platform()},
    "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                 "sklearn": sklearn.__version__, "xgboost": xgb.__version__, "lightgbm": lgb.__version__, "torch": TORCH_VERSION},
    "total_elapsed_seconds": secs_since(RUN_START),
}
for name, obj in [("eda_summary.json", eda_summary), ("cleaning_log.json", cleaning_log), ("data_dictionary.json", data_dictionary),
                  ("run_metadata.json", run_meta), ("insights.json", insights)]:
    with open(os.path.join(OUT, name), "w") as f:
        json.dump(obj, f, indent=2, default=str)
print("exports written:", sorted(os.listdir(OUT)))

In [8]:
# Cell 8: create the zip bundle and show the download link
from IPython.display import FileLink, HTML, Javascript, display

stamp = time.strftime("%Y%m%d_%H%M%S")
zip_name = f"demand_forecast_bundle_{stamp}.zip"
zip_path = os.path.join(WORK, zip_name)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUT):
        for fn in files:
            full = os.path.join(root, fn)
            zf.write(full, arcname=os.path.relpath(full, OUT))

print("zip created:", zip_path, "size MB:", round(os.path.getsize(zip_path) / 1e6, 2))
print("files inside:", len(zipfile.ZipFile(zip_path).namelist()))

display(FileLink(zip_name))
display(HTML(f'<a href="{zip_name}" download="{zip_name}">Click here to download {zip_name}</a>'))
display(Javascript(
    "setTimeout(function(){var a=document.createElement('a');"
    f"a.href='{zip_name}';a.download='{zip_name}';"
    "document.body.appendChild(a);a.click();},800);"
))

zip created: /kaggle/working/demand_forecast_bundle_20261110_171731.zip size MB: 13.47
files inside: 20


/kaggle/working/demand_forecast_bundle_20261110_171731.zip

<IPython.core.display.Javascript object>